# 第 1 章 解析傅里叶理论复习 · 代码实战（Python）

> 本书章代码的 Python 实现，逐段中文注释；对应 MATLAB 版本见 `matlab/fourier/ch1/`。

## basic · functions

`ch1_basic_functions.py`

In [ ]:
# ===== 书 1.5 节：基本函数（复现原书图 1.1 / 图 1.2） =====
# 这些函数是全书建模"孔径 / 光场"的积木，定义必须与原书表 1.2 一致。
# MATLAB/Python 差异：numpy 索引从 0 开始（MATLAB 从 1 开始）；
# np.meshgrid 默认 indexing='xy' 与 MATLAB meshgrid 行为一致，
# 对圆对称函数（circ、高斯）两种索引结果相同。
import numpy as np
import matplotlib.pyplot as plt


# ---------- 一维基本函数（表 1.2） ----------
def rect(x):
    """rect(x)：|x| < 1/2 取 1，|x| = 1/2 取 1/2，其余取 0。"""
    x = np.asarray(x, dtype=float)
    r = np.where(np.abs(x) < 0.5, 1.0, 0.0)
    r = np.where(np.abs(x) == 0.5, 0.5, r)   # 边界点取半值；均匀网格很少恰好命中
    return r


def tri(x):
    """Λ(x)：|x| <= 1 取 1-|x|，其余取 0。"""
    x = np.asarray(x, dtype=float)
    return np.where(np.abs(x) <= 1.0, 1.0 - np.abs(x), 0.0)


def sinc(x):
    """sinc(x) = sin(pi*x)/(pi*x)；x=0 处极限为 1。
    注意 numpy 自带的 np.sinc 就是这个归一化定义，等价于本函数。"""
    x = np.asarray(x, dtype=float)
    out = np.ones_like(x)
    mask = x != 0
    out[mask] = np.sin(np.pi * x[mask]) / (np.pi * x[mask])
    return out


def gauss(x):
    """高斯 exp(-pi*x^2)： Fourier 变换恰好是自己（表 1.3）。"""
    return np.exp(-np.pi * np.asarray(x, dtype=float) ** 2)


def circle(X, Y, R=0.5):
    """circ(sqrt(X^2+Y^2))：半径 R 的圆盘（表 1.2 中 R = 1/2）。
    注意：原书表 1.2 定义半径为 1/2，而表 1.3 的变换对对应半径 a 的圆盘，
    两者相差 2 倍半径——见 ch1_verify_circ_jinc.py 的数值验证。"""
    return np.where(np.hypot(X, Y) < R, 1.0, 0.0)


# ---------- 自检：定义是否与表 1.2 一致 ----------
print("rect(0)  =", rect(0.0), "(应为 1.0)")
print("rect(0.5)=", rect(0.5), "(应为 0.5)")
print("tri(0.5) =", tri(0.5), "(应为 0.5)")
print("sinc(0)  =", sinc(0.0), "(应为 1.0)")
print("sinc(0.5)=", sinc(0.5), "(应为 2/pi =", 2 / np.pi, ")")

# ---------- 复现原书图 1.1：一维基本函数 ----------
x = np.linspace(-1, 1, 801)
xw = np.linspace(-4, 4, 1201)

fig = plt.figure(figsize=(8, 9))
ax = fig.add_subplot(3, 2, 1)
ax.plot(x, rect(x)); ax.set_title("rect(x)"); ax.grid(True, alpha=0.3)
ax = fig.add_subplot(3, 2, 2)
ax.plot(xw, sinc(xw)); ax.set_title("sinc(x)"); ax.grid(True, alpha=0.3)
ax = fig.add_subplot(3, 2, 3)
ax.plot(x, tri(x)); ax.set_title(r"$\Lambda(x)$"); ax.grid(True, alpha=0.3)
ax = fig.add_subplot(3, 2, 4)
# comb 是 delta 周期阵列，网格上无法直接画，用 stem 箭头示意
xn = np.arange(-2, 3)
ax.stem(xn, np.ones_like(xn), basefmt="none")
ax.set_title("comb(x)"); ax.set_xlim(-2.5, 2.5); ax.set_ylim(0, 1.2)
ax = fig.add_subplot(3, 2, 5)
xg = np.linspace(-2, 2, 801)
ax.plot(xg, gauss(xg)); ax.set_title(r"$\exp(-\pi x^2)$"); ax.grid(True, alpha=0.3)
fig.suptitle("Basic 1D functions (reproduction of book Fig. 1.1)")
fig.tight_layout()
fig.savefig(__file__.replace("ch1_basic_functions.py", "out_1d_functions.png"), dpi=150)

# ---------- 复现原书图 1.2：二维 circ 与高斯 ----------
N = 80
xs = (np.arange(N) - N / 2) * (4.0 / N)
X, Y = np.meshgrid(xs, xs)               # 'xy' 索引，与 MATLAB meshgrid 一致
C = circle(X, Y, R=0.5)                  # 表 1.2：半径 1/2
G2 = np.exp(-np.pi * (X**2 + Y**2))      # 表 1.3 最后一类二维高斯（a=b=1）

fig = plt.figure(figsize=(10, 4.2))
ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.plot_wireframe(X, Y, C, rstride=2, cstride=2, color="k", linewidth=0.4)
ax.set_title(r"$\mathrm{circ}(\sqrt{x^2+y^2})$")
ax = fig.add_subplot(1, 2, 2, projection="3d")
ax.plot_wireframe(X, Y, G2, rstride=2, cstride=2, color="k", linewidth=0.4)
ax.set_title(r"$\exp[-\pi(x^2+y^2)]$")
fig.suptitle("Examples of 2D functions (reproduction of book Fig. 1.2)")
fig.tight_layout()
fig.savefig(__file__.replace("ch1_basic_functions.py", "out_2d_functions.png"), dpi=150)
print("已保存 out_1d_functions.png / out_2d_functions.png")


## 线性系统 · 演示

`ch1_lsi_demo.py`

In [ ]:
# ===== 书 1.6 节：线性空间不变系统 —— 卷积定理数值演示（式 1.11-1.13） =====
# 线性空间不变系统的输出是卷积：  g2(x,y) = (g1 * h)(x,y)
# 两条等价路线：
#   A. 直接卷积（对应式 1.11 的积分，物理直观）
#   B. 频域法（式 1.13）：FFT(g1) * H -> IFFT，H 是传递函数
# 本脚本用"零填充 + FFT"实现线性卷积，与 convolve2d 的直接结果对比，
# 两者应只差浮点舍入（~1e-16）。
# 注意：不填充直接用 fft2 相乘得到的是"循环卷积"，边界会互相缠绕——
# 这是第 2 章"计算以外的世界"和第 5 章角谱法的核心话题，这里先埋个伏笔。
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import convolve2d

N = 256            # 物面网格每边点数
L = 1.0            # 物面视场 [m]
dx = L / N
x = (np.arange(N) - N // 2) * dx
X, Y = np.meshgrid(x, x)

# ---------- 输入物 g1：圆盘 + 小方块 + 双缝（简单"物体"） ----------
g1 = np.where(np.hypot(X + 0.25, Y + 0.22) < 0.10, 1.0, 0.0)      # 圆盘
g1 += np.where((np.abs(X - 0.22) < 0.05) & (np.abs(Y + 0.20) < 0.05), 1.0, 0.0)  # 方块
g1 += np.where((np.abs(X - 0.05) < 0.012) & (np.abs(Y - 0.25) < 0.10), 1.0, 0.0)  # 细棒 1
g1 += np.where((np.abs(X + 0.05) < 0.012) & (np.abs(Y - 0.25) < 0.10), 1.0, 0.0)  # 细棒 2
g1 = np.clip(g1, 0, 1)

# ---------- 冲激响应 h：小圆盘（"模糊斑"，即光学里的点扩散函数 PSF） ----------
rh = 0.03
Kh = 31                       # h 网格边长（奇数，中心对齐）
xh = (np.arange(Kh) - Kh // 2) * dx
Xh, Yh = np.meshgrid(xh, xh)
h = np.where(np.hypot(Xh, Yh) < rh, 1.0, 0.0)
h /= h.sum()                  # 归一化能量：直流增益 H(0,0)=1，像不整体变亮

# ---------- 路线 B：频域法（零填充到 N+Kh-1，做线性卷积） ----------
P = N + Kh - 1
G2freq = np.fft.ifft2(np.fft.fft2(g1, s=(P, P)) * np.fft.fft2(h, s=(P, P)))
g2_freq = G2freq[(Kh // 2):(Kh // 2) + N, (Kh // 2):(Kh // 2) + N].real
# 裁切说明：h 中心在 (0,0)，其样本平移了 Kh//2；线性卷积结果里
# g1 的第 u 个样本对应整体输出的第 u + Kh//2 个。

# ---------- 路线 A：直接二维卷积（scipy 实现式 1.11 的求和） ----------
g2_direct = convolve2d(g1, h, mode="same", boundary="fill")

diff = np.max(np.abs(g2_direct - g2_freq))
rel = np.abs(g2_direct - g2_freq) / g2_freq.max()   # 相对误差（显示用）
print(f"两种路线结果最大差异 = {diff:.3e}（相对量 {rel.max():.1e}，浮点舍入量级）")
print(f"g2 峰值 = {g2_freq.max():.4f}（模糊后仍 > 单点峰值，因为物有连通结构）")

# ---------- 画图 ----------
fig, axes = plt.subplots(1, 4, figsize=(15, 3.9))
extent = [x[0], x[-1], x[0], x[-1]]
h_full = np.zeros_like(g1)               # 把 h 镶进全画面，统一显示比例
o = (N - Kh) // 2
h_full[o:o + Kh, o:o + Kh] = h
axes[0].imshow(g1, cmap="gray", extent=extent, origin="lower")
axes[0].set_title("input  g1 (object)")
axes[1].imshow(h_full, cmap="gray", extent=extent, origin="lower")
axes[1].set_title(f"impulse response h (PSF), {Kh}x{Kh} grid")
axes[2].imshow(g2_freq, cmap="gray", extent=extent, origin="lower")
axes[2].set_title("output  g2 = g1 * h")
im = axes[3].imshow(rel, extent=extent, origin="lower")
axes[3].set_title(f"|direct - spectral| / max(g2)\n(max {rel.max():.0e})")
plt.colorbar(im, ax=axes[3], fraction=0.046)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Linear space-invariant system: convolution vs. transfer-function route")
fig.tight_layout(rect=(0, 0, 1, 0.92))
fig.savefig(__file__.replace("ch1_lsi_demo.py", "out_lsi_demo.png"), dpi=150)
print("已保存 out_lsi_demo.png")


## 数值验证 · circ · jinc

`ch1_verify_circ_jinc.py`

In [ ]:
# ===== 书 1.5 节：circ 的二维傅里叶变换对验证（原书图 1.3 的 "jinc" 曲面） =====
# 表 1.2 定义 circ(sqrt(x^2+y^2)) 在半径 1/2 内取 1；
# 表 1.3 却给出 circ(r/a) <-> a^2*J1(2*pi*a*rho)/(a*rho)，峰值 pi*a^2（a=1 时为 pi，
# 正是图 1.3 标注的峰值）。半径 a/2 的圆盘面积是 pi*a^2/4，变换峰值应等于面积
# （中心纵坐标定理），所以表 1.3 这一对应的是"半径 a 的圆盘"（Goodman 约定）。
# 下面用二维 DFT 同时验证两种半径，看解析式到底匹配哪一个。
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import j1

N = 512            # 每边采样点数
L = 4.0            # 空间窗口宽度 [m]
dx = L / N
n = np.arange(N) - N // 2
x = n * dx
f = n / (N * dx)                 # 频率轴 [cycles/m]
X, Y = np.meshgrid(x, x)         # 'xy' 索引（与 MATLAB meshgrid 一致）；圆对称下无所谓


def disk(R):
    """半径 R 的圆盘（边界测度为零，取值不影响数值积分）。"""
    return np.where(np.hypot(X, Y) < R, 1.0, 0.0)


def ft2_centered(g):
    """二维连续 FT 的 DFT 近似（居中输出）：G ≈ dx^2 * Σ g exp(-j2π(f·r))。"""
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(g))) * dx**2


def jinc(rho, R):
    """半径 R 圆盘的解析频谱：R*J1(2*pi*R*rho)/rho，rho=0 处极限 = 圆盘面积。"""
    out = np.empty_like(rho)
    nz = rho != 0
    out[nz] = R * j1(2 * np.pi * R * rho[nz]) / rho[nz]
    out[~nz] = np.pi * R**2
    return out


results = {}
for R in (1.0, 0.5):
    G = ft2_centered(disk(R))
    row = G[N // 2, :].real       # fY = 0 的径向剖面（圆对称，行/列等价）
    A = jinc(f, R)
    err = np.max(np.abs(row - A))
    area = np.sum(disk(R)) * dx**2          # 中心纵坐标定理：G(0,0) = 面积
    results[R] = (f, row, A, err)
    print(f"R = {R:>4}: 频谱峰值(数值) = {row[N//2]:.6f}, 解析极限 = {np.pi*R**2:.6f}, "
          f"圆盘面积(数值) = {area:.6f}, 径向剖面最大误差 = {err:.3e}")

print("\n结论：解析式 R*J1(2*pi*R*rho)/rho 精确匹配'半径 R 的圆盘'。")
print("表 1.3（a=1，峰值 pi）对应半径 1 的圆盘；若坚持表 1.2 的半径 1/2 定义，")
print("变换应为 0.5*J1(pi*rho)/rho（峰值 pi/4）——两处约定不一致，使用时务必小心。")

# ---------- 画径向剖面对比图 ----------
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, R, ttl in [
    (axes[0], 1.0, "R = 1  (peak $\\pi$, matches Table 1.3 / Fig. 1.3)"),
    (axes[1], 0.5, "R = 1/2 (peak $\\pi/4$, matches Table 1.2 definition)"),
]:
    f_, row, A, err = results[R]
    m = np.abs(f_) <= 3.0
    ax.plot(f_[m], A[m], "b-", lw=1.5, label="analytic  $R\\,J_1(2\\pi R\\rho)/\\rho$")
    ax.plot(f_[m], row[m], "ro", ms=4, mfc="none", label="2D DFT numeric")
    ax.axhline(0, color="gray", lw=0.5)
    ax.set_title(f"{ttl}\nmax error = {err:.1e}")
    ax.set_xlabel(r"$\rho$ [cycles/m]"); ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle("Circle function transform (jinc) — radial cut fY = 0")
fig.tight_layout()
fig.savefig(__file__.replace("ch1_verify_circ_jinc.py", "out_circ_jinc.png"), dpi=150)
print("已保存 out_circ_jinc.png")


## 数值验证 · pairs · 1d

`ch1_verify_pairs_1d.py`

In [ ]:
# ===== 书 1.5 节：用 numpy.fft 数值验证表 1.3 的一维变换对 =====
# 连续傅里叶变换的 DFT 近似（矩形求和）：
#     G(f) ≈ dx * Σ_n g(x_n) exp(-j2π f x_n)
# numpy.fft 的正变换不除 N（归一化在逆变换里），因此乘 dx 就回到连续 FT 的量纲。
# 信号定义在 x ∈ [-L/2, L/2) 上（0-based 索引），标准"居中"流程：
#     ifftshift -> fft -> fftshift
# 得到按频率从负到正排列的频谱（本书第 2、3 章系统讲这套定标）。
# MATLAB 注意：对偶数 N，fftshift 与 ifftshift 等价；MATLAB fft 同样不除 N。
import numpy as np
import matplotlib.pyplot as plt

N = 512            # 采样点数（偶数）
L = 8.0            # 空间窗口宽度 [m]
dx = L / N         # 采样间隔 [m]
n = np.arange(N) - N // 2
x = n * dx                       # x ∈ [-L/2, L/2)
f = n / (N * dx)                 # 频率轴 [cycles/m]，与 x 同序（居中）


def rect(t):
    """rect：|t|<1/2 为 1，|t|=1/2 为 1/2，其余 0（表 1.2）。"""
    t = np.asarray(t, dtype=float)
    r = np.where(np.abs(t) < 0.5, 1.0, 0.0)
    return np.where(np.abs(t) == 0.5, 0.5, r)


def tri(t):
    """Λ：|t|<=1 为 1-|t|，其余 0（表 1.2）。"""
    t = np.asarray(t, dtype=float)
    return np.where(np.abs(t) <= 1.0, 1.0 - np.abs(t), 0.0)


def ft_centered(g):
    """一维连续 FT 的 DFT 近似（居中输出）。"""
    return np.fft.fftshift(np.fft.fft(np.fft.ifftshift(g))) * dx


def sincn(t):
    """归一化 sinc：sin(pi t)/(pi t)，与 numpy.sinc / MATLAB sinc 一致。"""
    return np.sinc(t)


# ---------- 对 1：rect(x) <-> sinc(f)（表 1.3 第 1 行，a=1） ----------
G1 = ft_centered(rect(x))
e1 = np.max(np.abs(G1 - sincn(f)))
print(f"[1] rect(x) <-> sinc(f)          最大绝对误差 = {e1:.3e}")

# ---------- 对 2：Λ(x) <-> sinc^2(f)（表 1.3 第 3 行，a=1） ----------
G2 = ft_centered(tri(x))
e2 = np.max(np.abs(G2 - sincn(f) ** 2))
print(f"[2] tri(x)  <-> sinc^2(f)        最大绝对误差 = {e2:.3e}")

# ---------- 对 3：exp(-pi x^2) <-> exp(-pi f^2)（表 1.3 第 5 行，a=1） ----------
g3 = np.exp(-np.pi * x**2)
G3 = ft_centered(g3)
e3 = np.max(np.abs(G3 - np.exp(-np.pi * f**2)))
print(f"[3] exp(-pi x^2) <-> exp(-pi f^2) 最大绝对误差 = {e3:.3e}")

# ---------- 定理 1：相似性 rect(x/2) <-> 2 sinc(2f)（表 1.1，a=2） ----------
G4 = ft_centered(rect(x / 2))
e4 = np.max(np.abs(G4 - 2 * sincn(2 * f)))
print(f"[4] 相似性 rect(x/2) <-> 2sinc(2f) 最大绝对误差 = {e4:.3e}")

# ---------- 定理 2：位移 rect(x-0.7) <-> sinc(f) exp(-j2*pi*f*0.7)（表 1.1） ----------
x0 = 0.7
G5 = ft_centered(rect(x - x0))
e5 = np.max(np.abs(G5 - sincn(f) * np.exp(-2j * np.pi * f * x0)))
print(f"[5] 位移   rect(x-0.7) 复数最大误差 = {e5:.3e}")

# ---------- 定理 3：中心纵坐标 G(0) = ∫g dx（表 1.1 最后一行） ----------
sums = np.sum(rect(x)) * dx
print(f"[6] 中心纵坐标  G(0): 数值 = {G1[N//2].real:.6f}, "
      f"求和*dx = {sums:.6f}, 解析 = 1.0")

# ---------- 勘误验证：表 1.3 最后一行 chirp 变换对（符号有误） ----------
# 原书：exp(-j*pi*x^2) <-> j*exp(-j*pi*f^2)（2D 版本 j*|ab|*exp[-j*pi(...)]）
# 正确：exp(-j*pi*x^2) <-> exp(-j*pi/4)*exp(+j*pi*f^2)
#       （2D 版本 -j*|ab|*exp[+j*pi(a^2 fX^2 + b^2 fY^2)]）
# chirp 不衰减，截断积分误差 ~ 1/(窗口半宽)，用宽窗口就能清晰区分两种公式。
Nc = 4096
Lc = 40.0
dxc = Lc / Nc
nc = np.arange(Nc) - Nc // 2
xc = nc * dxc
fc = nc / (Nc * dxc)
gc = np.exp(-1j * np.pi * xc**2)
Gc = np.fft.fftshift(np.fft.fft(np.fft.ifftshift(gc))) * dxc
m = np.abs(fc) <= 1.0
mine = np.exp(-1j * np.pi / 4) * np.exp(1j * np.pi * fc**2)
book = 1j * np.exp(-1j * np.pi * fc**2)
print(f"[7] chirp 修正式误差 = {np.max(np.abs(Gc[m] - mine[m])):.3e}, "
      f"原书式误差 = {np.max(np.abs(Gc[m] - book[m])):.3e}  （应前者 << 后者）")

# ---------- 画对比图：数值（圆点） vs 解析（实线） ----------
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, G, A, ttl in [
    (axes[0], G1, sincn(f), r"rect $\leftrightarrow$ sinc"),
    (axes[1], G2, sincn(f) ** 2, r"tri $\leftrightarrow$ sinc$^2$"),
    (axes[2], G3, np.exp(-np.pi * f**2), r"gauss $\leftrightarrow$ gauss"),
]:
    m = np.abs(f) <= 4
    ax.plot(f[m], A[m], "b-", lw=1.5, label="analytic")
    ax.plot(f[m], G[m].real, "ro", ms=4, mfc="none", label="DFT numeric")
    ax.set_title(ttl); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
    ax.set_xlabel("f [cycles/m]")
fig.suptitle(f"Table 1.3 pairs verified with numpy.fft (N={N}, L={L}, "
             f"max errors {e1:.1e} / {e2:.1e} / {e3:.1e})")
fig.tight_layout()
fig.savefig(__file__.replace("ch1_verify_pairs_1d.py", "out_1d_pairs.png"), dpi=150)
print("已保存 out_1d_pairs.png")
